In [1]:
from langchain_ollama import ChatOllama

In [2]:
llm = ChatOllama(
    model="llama3.2:latest"
)

In [5]:
result = llm.invoke("hi")

In [6]:
result.content

'How can I help you today?'

In [8]:
from langchain_community.vectorstores import FAISS

FAISS_SAVE_PATH = "faiss_skin_lesion_store"

In [9]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
)

C:\Users\sjuna\AppData\Local\Temp\ipykernel_14520\9015449.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


In [10]:
faiss_store = FAISS.load_local(
        FAISS_SAVE_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )

In [11]:
retriever = faiss_store.as_retriever(search_kwargs={"k": 6,})

In [12]:
from langchain_core.prompts import ChatPromptTemplate


medical_report_prompt = ChatPromptTemplate.from_messages(
    [
        # SYSTEM MESSAGE
        (
            "system",
            """
You are a medical assistant specializing in dermatology.
You have the knowledge and communication style of an experienced dermatologist.

IMPORTANT RULES:
- Use ONLY the information provided in the Retrieved Medical Context.
- Do NOT add external medical knowledge.
- Do NOT make assumptions beyond the given data.
- Do NOT provide treatment, medication, or cure advice.
- Do NOT claim a definitive diagnosis.
- Always communicate in a calm, professional, and reassuring manner.
- If information is missing, clearly state that it is not available in the provided context.

Your role is to EXPLAIN findings, not to replace a doctor.
"""
        ),

        # HUMAN MESSAGE
        (
            "human",
            """
Retrieved Medical Context:
{context}

Patient Information:
- Predicted Skin Lesion Type: {predicted_class}
- Model Confidence Score: {confidence}%
- Explainability Summary (Grad-CAM / Heatmap):
{gradcam_summary}

Report Type:
{report_type}

Task:
Explain the result as an experienced dermatologist would.

Guidelines:
1. Clearly explain what the detected skin lesion means.
2. Explain the confidence score in simple terms.
3. Describe why the highlighted image regions are important.
4. Use simple, non-alarming language understandable to a non-medical person.
5. Mention the general risk level if present in the context.
6. Encourage professional medical consultation without giving advice or treatment.
7. If uncertainty exists, state it clearly and responsibly.

Response Style Rules:
- If Report Type is "Doctor":
  * Use precise medical terminology
  * Maintain a clinical and professional tone

- If Report Type is "Patient":
  * Use simple everyday language
  * Avoid medical jargon
  * Be reassuring and explanatory
"""
        )
    ]
)

In [19]:
from pydantic import BaseModel,Field

class report(BaseModel):
    skin_lesion_type: str = Field(..., description="name of the detected skin_lesion_type")
    model_confidence_score: float = Field(..., description="models confidence score")
    gradcam_summary: str = Field(..., description="gradcam summary")

In [20]:
model_with_structure = llm.with_structured_output(report)

In [21]:
rag_chain = medical_report_prompt | model_with_structure

In [22]:
response = rag_chain.invoke({
    "context": retriever,
    "predicted_class": "Melanocytic Nevus",
    "confidence": 92.4,
    "gradcam_summary": "Model focused on pigmentation and shape irregularities",
    "report_type": "Patient"
})

In [31]:
details = response.model_dump()
for msg in details.items():
    print(msg)

('skin_lesion_type', 'Melanocytic Nevus')
('model_confidence_score', 92.4)
('gradcam_summary', 'The model is focusing on the pigmentation and shape irregularities of the skin lesion.')


In [7]:
from langchain_community.document_loaders import PyPDFLoader

file_path = r"D:\Artificial Intelligence\College project\oxford_dermatology.pdf"
loader = PyPDFLoader(file_path)

In [8]:
doc = loader.load()

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(doc)

In [20]:
len(chunks)

1550

In [21]:
chunks[0]

Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-01-27T06:56:31+00:00', 'source': 'D:\\Artificial Intelligence\\College project\\oxford_dermatology.pdf', 'total_pages': 590, 'page': 0, 'page_label': '1'}, page_content='8 CHA pt Er\xa01  Structure and function of\xa0the\xa0skin\nDermis and\xa0glands\nthe dermis is a layer of connective tissue beneath the epidermis. A\xa0layer \nof subcutaneous fat separates the dermis from the underlying fascia \nand muscle. t he dermis has a rich supply of blood vessels, lymphatics, \nnerves, and sensory receptors. t he thickness of the dermis varies with \nbody site and may measure as much as 5mm on the\xa0back.\nStructural components\n• Collagen, mainly type I\xa0but some type III, gives the dermis tensile \nstrength. Ageing skin is characterized by reduced collagen synthesis \nand increased collagen breakdown by matrix metalloproteinases.\n• Elastic fibres, containing a core of elastin, supply elast

In [22]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
)

In [24]:
from langchain_community.vectorstores import FAISS

In [25]:
db = FAISS.from_documents(chunks,embeddings)

In [26]:
retriever = db.as_retriever(search_kwargs={"k": 6,})

In [29]:
from langchain_core.prompts import ChatPromptTemplate

medical_report_prompt = ChatPromptTemplate.from_messages(
    [
        # SYSTEM MESSAGE
        (
            "system",
            """
You are a medical assistant specializing in dermatology.
You have the knowledge and communication style of an experienced dermatologist.

IMPORTANT RULES:
- Use ONLY the information provided in the Retrieved Medical Context.
- Do NOT add external medical knowledge.
- Do NOT make assumptions beyond the given data.
- Do NOT provide treatment, medication, or cure advice.
- Do NOT claim a definitive diagnosis.
- Always communicate in a calm, professional, and reassuring manner.
- If information is missing, clearly state that it is not available in the provided context.

Your role is to EXPLAIN findings, not to replace a doctor.
"""
        ),

        # HUMAN MESSAGE
        (
            "human",
            """
Retrieved Medical Context:
{context}

Patient Information:
- Predicted Skin Lesion Type: {predicted_class}
- Model Confidence Score: {confidence}%
- Explainability Summary (Grad-CAM / Heatmap):
{gradcam_summary}

Report Type:
{report_type}

Task:
Explain the result as an experienced dermatologist would.

Guidelines:
1. Clearly explain what the detected skin lesion means.
2. Explain the confidence score in simple terms.
3. Describe why the highlighted image regions are important.
4. Use simple, non-alarming language understandable to a non-medical person.
5. Mention the general risk level if present in the context.
6. Encourage professional medical consultation without giving advice or treatment.
7. If uncertainty exists, state it clearly and responsibly.

Response Style Rules:
- If Report Type is "Doctor":
  * Use precise medical terminology
  * Maintain a clinical and professional tone

- If Report Type is "Patient":
  * Use simple everyday language
  * Avoid medical jargon
  * Be reassuring and explanatory
"""
        )
    ]
)


In [30]:
rag_chain = medical_report_prompt | llm

In [32]:
response = rag_chain.invoke({
    "context": retriever,
    "predicted_class": "Melanocytic Nevus",
    "confidence": 92.4,
    "gradcam_summary": "Model focused on pigmentation and shape irregularities",
    "report_type": "Patient"
})

In [34]:
response.content

"**Report for [Patient's Name]**\n\nI've reviewed the results of our skin lesion analysis, and I'd like to explain what we found.\n\nThe detected skin lesion is a **Melanocytic Nevus**, which is a type of benign growth. A Melanocytic Nevus is a common condition where a mole or patch of skin grows in an abnormal shape and color. Don't worry; most Melanocytic Nevi are harmless and don't require treatment.\n\nThe model's confidence score of 92.4% indicates that our analysis was very accurate. This means the model was able to identify the lesion as a Melanocytic Nevus with high reliability.\n\nNow, let's take a closer look at the **Grad-CAM/Heatmap** provided. The model focused on two key areas: **pigmentation irregularities** and **shape abnormalities**. These regions are important because they help us understand how the skin lesion differs from surrounding healthy tissue.\n\nThe highlighted image regions show where the skin lesion is most different in color and shape compared to normal s